In [2]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import load_img, img_to_array, save_img


2026-02-24 06:36:44.950751: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-24 06:36:45.066472: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-24 06:36:45.067343: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-24 06:36:45.733340: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [3]:
datagen = ImageDataGenerator(
    rotation_range=15,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

In [4]:
input_dir = "/home/mansi/Downloads/dataset (2)/dataset"
output_dir = "/home/mansi/Downloads/dataset (2)/augmented_dataset"

In [5]:
os.makedirs(output_dir, exist_ok=True)

In [6]:
augmentation_plan = {
    "broken": 1,
    "intact": 4
}


In [7]:
for class_name in ["broken", "intact"]:
    class_input_path = os.path.join(input_dir, class_name)
    class_output_path = os.path.join(output_dir, class_name)
    os.makedirs(class_output_path, exist_ok=True)

    for img_name in os.listdir(class_input_path):
        img_path = os.path.join(class_input_path, img_name)

        img = load_img(img_path)
        img_array = img_to_array(img)
        img_array = img_array.reshape((1,) + img_array.shape)

        # Save original
        save_img(os.path.join(class_output_path, img_name), img_array[0])

        num_aug = augmentation_plan[class_name]

        i = 0
        for batch in datagen.flow(img_array, batch_size=1):
            new_img_name = f"aug_{i}_{img_name}"
            save_img(os.path.join(class_output_path, new_img_name), batch[0])
            i += 1
            if i >= num_aug:
                break

In [8]:
import os
import shutil
import random

base_dir = "/home/mansi/Downloads/dataset (2)/augmented_dataset"

train_ratio = 0.7
val_ratio = 0.15
test_ratio = 0.15

for class_name in ["broken", "intact"]:
    class_path = os.path.join(base_dir, class_name)
    images = os.listdir(class_path)
    random.shuffle(images)

    total = len(images)
    train_end = int(total * train_ratio)
    val_end = int(total * (train_ratio + val_ratio))

    splits = {
        "train": images[:train_end],
        "val": images[train_end:val_end],
        "test": images[val_end:]
    }

    for split_name, split_images in splits.items():
        split_dir = os.path.join(base_dir, split_name, class_name)
        os.makedirs(split_dir, exist_ok=True)

        for img in split_images:
            src = os.path.join(class_path, img)
            dst = os.path.join(split_dir, img)
            shutil.copy(src, dst)

print("✅ Dataset successfully split!")

✅ Dataset successfully split!


In [9]:
import os
import shutil
import random

base_dir = "/home/mansi/Downloads/dataset (2)/augmented_dataset"

train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

for class_name in ["broken", "intact"]:
    class_path = os.path.join(base_dir, class_name)
    images = os.listdir(class_path)
    random.shuffle(images)

    total = len(images)
    train_end = int(total * train_ratio)
    val_end = int(total * (train_ratio + val_ratio))

    splits = {
        "train": images[:train_end],
        "val": images[train_end:val_end],
        "test": images[val_end:]
    }

    for split_name, split_images in splits.items():
        split_dir = os.path.join(base_dir, split_name, class_name)
        os.makedirs(split_dir, exist_ok=True)

        for img in split_images:
            src = os.path.join(class_path, img)
            dst = os.path.join(split_dir, img)
            shutil.copy(src, dst)

print("✅ Dataset successfully split (80/10/10)!")

✅ Dataset successfully split (80/10/10)!


In [10]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = 224
BATCH_SIZE = 32

base_path = "/home/mansi/Downloads/dataset (2)/augmented_dataset"

train_ds = tf.keras.utils.image_dataset_from_directory(
    base_path + "/train",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    base_path + "/val",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    base_path + "/test",
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE
)

Found 464 files belonging to 2 classes.
Found 58 files belonging to 2 classes.
Found 58 files belonging to 2 classes.


2026-02-24 07:23:01.673788: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-02-24 07:23:01.783315: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [11]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [12]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

9406464/9406464 [==============================] - 3s 0us/step


In [13]:
inputs = keras.Input(shape=(224, 224, 3))

x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x = base_model(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

In [14]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [15]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

Epoch 1/10
15/15 [==============================] - 5s 257ms/step - loss: 0.6505 - accuracy: 0.6810 - val_loss: 0.3167 - val_accuracy: 0.9138
Epoch 2/10
15/15 [==============================] - 4s 236ms/step - loss: 0.2818 - accuracy: 0.9116 - val_loss: 0.2007 - val_accuracy: 0.9310
Epoch 3/10
15/15 [==============================] - 3s 227ms/step - loss: 0.1639 - accuracy: 0.9569 - val_loss: 0.1613 - val_accuracy: 0.9483
Epoch 4/10
15/15 [==============================] - 4s 236ms/step - loss: 0.1286 - accuracy: 0.9612 - val_loss: 0.1423 - val_accuracy: 0.9483
Epoch 5/10
15/15 [==============================] - 4s 243ms/step - loss: 0.1054 - accuracy: 0.9677 - val_loss: 0.1312 - val_accuracy: 0.9483
Epoch 6/10
15/15 [==============================] - 4s 234ms/step - loss: 0.0938 - accuracy: 0.9849 - val_loss: 0.1299 - val_accuracy: 0.9483
Epoch 7/10
15/15 [==============================] - 3s 224ms/step - loss: 0.0860 - accuracy: 0.9828 - val_loss: 0.1142 - val_accuracy: 0.9483
Epoch 

In [16]:
test_loss, test_acc = model.evaluate(test_ds)
print("Test Accuracy:", test_acc)

2/2 [==============================] - 0s 191ms/step - loss: 0.1022 - accuracy: 0.9655
Test Accuracy: 0.9655172228813171


In [27]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

# Get true labels
y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images)
    preds = (preds > 0.5).astype(int)  # convert probabilities to 0/1

    y_true.extend(labels.numpy())
    y_pred.extend(preds.flatten())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

1/1 [==============================] - 0s 179ms/step


In [28]:
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[28  2]
 [ 0 28]]


In [29]:
print(classification_report(y_true, y_pred, target_names=["broken", "intact"]))

              precision    recall  f1-score   support

      broken       1.00      0.93      0.97        30
      intact       0.93      1.00      0.97        28

    accuracy                           0.97        58
   macro avg       0.97      0.97      0.97        58
weighted avg       0.97      0.97      0.97        58



In [1]:
################################

In [2]:
import os
import shutil
from sklearn.model_selection import train_test_split

In [3]:
base_dir = "/home/mansi/Downloads/dataset/dataset"
output_dir = "/home/mansi/Downloads/dataset/data_split"

classes = ["broken", "intact"]
for split in ["train", "val", "test"]:
    for cls in classes:
        os.makedirs(os.path.join(output_dir, split, cls), exist_ok=True)


In [4]:
for cls in classes:
    cls_path = os.path.join(base_dir, cls)
    images = os.listdir(cls_path)

    # 80% train, 20% temp
    train_imgs, temp_imgs = train_test_split(
        images, test_size=0.2, random_state=42
    )

    # Split 20% temp into 10% val and 10% test
    val_imgs, test_imgs = train_test_split(
        temp_imgs, test_size=0.5, random_state=42
    )

    for img in train_imgs:
        shutil.copy(os.path.join(cls_path, img),
                    os.path.join(output_dir, "train", cls, img))

    for img in val_imgs:
        shutil.copy(os.path.join(cls_path, img),
                    os.path.join(output_dir, "val", cls, img))

    for img in test_imgs:
        shutil.copy(os.path.join(cls_path, img),
                    os.path.join(output_dir, "test", cls, img))

print("✅ 80-10-10 Dataset split completed!")

✅ 80-10-10 Dataset split completed!


In [5]:
for split in ["train", "val", "test"]:
    for cls in classes:
        path = f"/home/mansi/Downloads/dataset/data_split/{split}/{cls}"
        print(split, cls, len(os.listdir(path)))

train broken 263
train intact 94
val broken 33
val intact 12
test broken 33
test intact 12


In [98]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# 🔹 Paths
train_dir = "/home/mansi/Downloads/dataset/data_split/train"
val_dir   = "/home/mansi/Downloads/dataset/data_split/val"
test_dir  = "/home/mansi/Downloads/dataset/data_split/test"

# 🔹 Image size & batch size
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

# ===============================
# 🟢 TRAIN GENERATOR (With Augmentation)
# ===============================

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # Important for MobileNet
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.75, 1.25],
    horizontal_flip=True,
    fill_mode='nearest'
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

# ===============================
# 🔵 VALIDATION GENERATOR (No Augmentation)
# ===============================

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

# ===============================
# 🔴 TEST GENERATOR (No Augmentation)
# ===============================

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

# ===============================
# 📊 Verify class indices
# ===============================

print("Class indices:", train_generator.class_indices)
print("Train samples:", train_generator.samples)
print("Validation samples:", val_generator.samples)
print("Test samples:", test_generator.samples)

Found 357 images belonging to 2 classes.
Found 45 images belonging to 2 classes.
Found 45 images belonging to 2 classes.
Class indices: {'broken': 0, 'intact': 1}
Train samples: 357
Validation samples: 45
Test samples: 45


In [8]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam

# Load pretrained MobileNetV2
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze base model
base_model.trainable = False

# Build model
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

# Compile
model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(),
             tf.keras.metrics.Recall()]
)

model.summary()

2026-03-03 19:38:21.656378: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:995] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-03-03 19:38:21.878085: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1960] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mobilenetv2_1.00_224 (Func  (None, 7, 7, 1280)        2257984   
 tional)                                                         
                                                                 
 global_average_pooling2d (  (None, 1280)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dense (Dense)               (None, 128)               163968    
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 1)                 129       
                                                                 
Total params: 2422081 (9.24 MB)
Trainable params: 164097

In [101]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3
)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/15


ValueError: in user code:

    File "/home/mansi/miniconda3/envs/venvfl/lib/python3.9/site-packages/keras/src/engine/training.py", line 1338, in train_function  *
        return step_function(self, iterator)
    File "/home/mansi/miniconda3/envs/venvfl/lib/python3.9/site-packages/keras/src/engine/training.py", line 1322, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "/home/mansi/miniconda3/envs/venvfl/lib/python3.9/site-packages/keras/src/engine/training.py", line 1303, in run_step  **
        outputs = model.train_step(data)
    File "/home/mansi/miniconda3/envs/venvfl/lib/python3.9/site-packages/keras/src/engine/training.py", line 1081, in train_step
        loss = self.compute_loss(x, y, y_pred, sample_weight)
    File "/home/mansi/miniconda3/envs/venvfl/lib/python3.9/site-packages/keras/src/engine/training.py", line 1139, in compute_loss
        return self.compiled_loss(
    File "/home/mansi/miniconda3/envs/venvfl/lib/python

In [102]:
# Unfreeze the base model
base_model.trainable = True

# Freeze all layers except last 30
for layer in base_model.layers[:-30]:
    layer.trainable = False

print("Total layers:", len(base_model.layers))

Total layers: 154


In [103]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=1e-5),  # lower LR
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(),
        tf.keras.metrics.Recall()
    ]
)

In [104]:
history_fine = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=[early_stop, reduce_lr]
)

Epoch 1/10


ValueError: in user code:

    File "/home/mansi/miniconda3/envs/venvfl/lib/python3.9/site-packages/keras/src/engine/training.py", line 1338, in train_function  *
        return step_function(self, iterator)
    File "/home/mansi/miniconda3/envs/venvfl/lib/python3.9/site-packages/keras/src/engine/training.py", line 1322, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "/home/mansi/miniconda3/envs/venvfl/lib/python3.9/site-packages/keras/src/engine/training.py", line 1303, in run_step  **
        outputs = model.train_step(data)
    File "/home/mansi/miniconda3/envs/venvfl/lib/python3.9/site-packages/keras/src/engine/training.py", line 1081, in train_step
        loss = self.compute_loss(x, y, y_pred, sample_weight)
    File "/home/mansi/miniconda3/envs/venvfl/lib/python3.9/site-packages/keras/src/engine/training.py", line 1139, in compute_loss
        return self.compiled_loss(
    File "/home/mansi/miniconda3/envs/venvfl/lib/python

In [105]:
test_loss, test_acc, test_prec, test_rec = model.evaluate(test_generator)

print("Test Accuracy:", test_acc)
print("Test Precision:", test_prec)
print("Test Recall:", test_rec)

ValueError: in user code:

    File "/home/mansi/miniconda3/envs/venvfl/lib/python3.9/site-packages/keras/src/engine/training.py", line 1972, in test_function  *
        return step_function(self, iterator)
    File "/home/mansi/miniconda3/envs/venvfl/lib/python3.9/site-packages/keras/src/engine/training.py", line 1956, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "/home/mansi/miniconda3/envs/venvfl/lib/python3.9/site-packages/keras/src/engine/training.py", line 1944, in run_step  **
        outputs = model.test_step(data)
    File "/home/mansi/miniconda3/envs/venvfl/lib/python3.9/site-packages/keras/src/engine/training.py", line 1852, in test_step
        self.compute_loss(x, y, y_pred, sample_weight)
    File "/home/mansi/miniconda3/envs/venvfl/lib/python3.9/site-packages/keras/src/engine/training.py", line 1139, in compute_loss
        return self.compiled_loss(
    File "/home/mansi/miniconda3/envs/venvfl/lib/python3.9/site-p

In [14]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

preds = model.predict(test_generator)
pred_labels = (preds > 0.5).astype(int)

cm = confusion_matrix(test_generator.classes, pred_labels)
print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(test_generator.classes, pred_labels))

3/3 [==============================] - 1s 98ms/step
Confusion Matrix:
[[33  0]
 [ 0 12]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        33
           1       1.00      1.00      1.00        12

    accuracy                           1.00        45
   macro avg       1.00      1.00      1.00        45
weighted avg       1.00      1.00      1.00        45



In [15]:
model.save("medcheck_mobilenet_v2.h5")

/home/mansi/miniconda3/envs/venvfl/lib/python3.9/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [16]:
model.save("medcheck_mobilenet_v2.keras")

In [17]:
model.save_weights("medcheck_weights.h5")

In [3]:
from tensorflow.keras.models import load_model

model = load_model("medcheck_mobilenet_v2.keras")

TypeError: Unable to convert function return value to a Python type! The signature was
	() -> handle

In [2]:
import os
import numpy as np
import cv2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

IMG_SIZE = 224
test_folder = "/home/mansi/Downloads/dataset/new_test"

def predict_image(image_path):
    img = cv2.imread(image_path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    img = np.array(img, dtype=np.float32)
    img = preprocess_input(img)

    img = np.expand_dims(img, axis=0)

    prediction = model.predict(img, verbose=0)[0][0]

    label = "INTACT" if prediction > 0.5 else "BROKEN"
    confidence = prediction if prediction > 0.5 else 1 - prediction

    return label, confidence

print("---- New Image Predictions ----\n")

for img_name in os.listdir(test_folder):
    img_path = os.path.join(test_folder, img_name)
    
    if img_name.lower().endswith((".jpg", ".jpeg", ".png")):
        label, conf = predict_image(img_path)
        print(f"{img_name} → {label} (Confidence: {conf:.4f})")

TypeError: Unable to convert function return value to a Python type! The signature was
	() -> handle

In [22]:
for layer in model.layers[0].layers:
    if "conv" in layer.name.lower():
        print(layer.name)

Conv1
bn_Conv1
Conv1_relu
expanded_conv_depthwise
expanded_conv_depthwise_BN
expanded_conv_depthwise_relu
expanded_conv_project
expanded_conv_project_BN
Conv_1
Conv_1_bn


In [89]:
import tensorflow as tf
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image

# 1. Standard Preprocessing
def get_img_array(img_path, size=(224, 224)):
    img = tf.keras.preprocessing.image.load_img(img_path, target_size=size)
    array = tf.keras.preprocessing.image.img_to_array(img)
    array = np.expand_dims(array, axis=0)
    return tf.keras.applications.mobilenet_v2.preprocess_input(array)

# 2. Grad-CAM logic
def make_gradcam_heatmap(img_array, model, last_conv_layer_name="out_relu"):
    try:
        grad_model = tf.keras.models.Model(
            inputs=[model.inputs],
            outputs=[model.get_layer(last_conv_layer_name).output, model.output]
        )
    except Exception:
        return np.zeros((7, 7), dtype=np.float32)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_index = tf.argmax(predictions[0])
        loss = predictions[:, pred_index]

    grads = tape.gradient(loss, conv_outputs)
    if grads is None:
        return np.zeros((7, 7), dtype=np.float32)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    
    # Weighted sum: $L_{Grad-CAM}^c = ReLU(\sum_k \alpha_k^c A^k)$
    heatmap = tf.reduce_sum(tf.multiply(pooled_grads, conv_outputs), axis=-1)

    heatmap = tf.maximum(heatmap, 0)
    max_val = tf.math.reduce_max(heatmap).numpy()
    if max_val == 0: max_val = 1e-10
    heatmap = heatmap / max_val

    # Convert to pure NumPy array to strip TF metadata
    return np.array(heatmap.numpy().tolist(), dtype=np.float32)

# 3. Manual Overlay (Modernized to remove Deprecation Warnings)
def save_gradcam_result(img_path, heatmap, save_path):
    # Load original image
    img = Image.open(img_path).convert('RGB')
    width, height = img.size
    
    # Convert heatmap to PIL and resize (Removed deprecated 'mode')
    heatmap_uint8 = np.uint8(255 * heatmap)
    heatmap_img = Image.fromarray(heatmap_uint8)
    heatmap_img = heatmap_img.resize((width, height), resample=Image.BILINEAR)
    
    # Apply colormap (Updated to modern Matplotlib 3.7+ syntax)
    colormap = plt.get_cmap('jet')
    
    heatmap_np = np.array(heatmap_img) / 255.0
    heatmap_colored = colormap(heatmap_np)[:, :, :3] # Extract RGB, ignore Alpha
    heatmap_colored = np.uint8(heatmap_colored * 255)
    
    # Blend: 60% Original + 40% Heatmap
    img_np = np.array(img)
    overlayed = (img_np * 0.6 + heatmap_colored * 0.4).astype(np.uint8)
    
    # Save
    Image.fromarray(overlayed).save(save_path)

# 4. Main Loop
TEST_DIR = "/home/mansi/Downloads/dataset/new_test"
RESULT_DIR = "gradcam_results"
os.makedirs(RESULT_DIR, exist_ok=True)

image_files = [f for f in os.listdir(TEST_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

for filename in image_files:
    img_path = os.path.join(TEST_DIR, filename)
    save_path = os.path.join(RESULT_DIR, filename)
    
    try:
        img_array = get_img_array(img_path)
        heatmap = make_gradcam_heatmap(img_array, model, "out_relu")
        save_gradcam_result(img_path, heatmap, save_path)
        print(f"✅ Processed: {filename}")
            
    except Exception as e:
        print(f"❌ Error on {filename}: {str(e)}")

✅ Processed: b-7.png
✅ Processed: i-4.png
✅ Processed: b-10.png
✅ Processed: b-3.png
✅ Processed: b-9.png
✅ Processed: i-6.png
✅ Processed: b-2.png
✅ Processed: i-2.png
✅ Processed: b-8.png
✅ Processed: b-11.png
✅ Processed: i-3.png
✅ Processed: b-1.png
✅ Processed: b-6.png
✅ Processed: b-5.png
✅ Processed: i-1.png
✅ Processed: b-4.png


In [92]:
import os
import math
from PIL import Image, ImageDraw

def create_pil_grid(folder_name="gradcam_results", cols=4, save_name="summary_grid.png"):
    # 1. Get the absolute path to ensure no confusion
    base_path = os.getcwd()
    result_dir = os.path.join(base_path, folder_name)
    
    if not os.path.exists(result_dir):
        print(f"❌ Error: The folder '{result_dir}' does not exist!")
        return

    # 2. Get valid images only
    valid_extensions = ('.png', '.jpg', '.jpeg')
    images = [f for f in os.listdir(result_dir) if f.lower().endswith(valid_extensions)]
    images.sort()

    if not images:
        print(f"❌ No images found in: {result_dir}")
        return

    print(f"🖼️ Found {len(images)} images. Stitching them together...")

    # 3. Setup Dimensions based on the first image
    with Image.open(os.path.join(result_dir, images[0])) as sample_img:
        w, h = sample_img.size
    
    num_images = len(images)
    rows = math.ceil(num_images / cols)

    # 4. Create a large blank canvas
    grid_w = cols * w
    grid_h = rows * h
    grid_img = Image.new('RGB', (grid_w, grid_h), color=(255, 255, 255))

    # 5. Paste images
    for i, img_name in enumerate(images):
        img_path = os.path.join(result_dir, img_name)
        with Image.open(img_path) as img:
            row = i // cols
            col = i % cols
            grid_img.paste(img, (col * w, row * h))
            
            # Label the images so you know which is which
            draw = ImageDraw.Draw(grid_img)
            draw.text((col * w + 10, row * h + 10), img_name, fill=(255, 255, 255))

    # 6. Save to your desktop/current folder
    grid_img.save(save_name)
    print(f"✅ Success! Grid saved at: {os.path.join(base_path, save_name)}")

# Run it
create_pil_grid("gradcam_results")

🖼️ Found 16 images. Stitching them together...
✅ Success! Grid saved at: /home/mansi/Desktop/mini_project/summary_grid.png


In [93]:
#############################